# Zarya Digital Bank: Week 10 — Day 2 Dual SFT Fine-Tuning Pipeline
**Objective:** Interactive Supervised Fine-Tuning (SFT) & Ablation Pipeline for **Qwen 2.5 3B** and **Llama 3.2 3B** using 4-bit QLoRA and Unsloth on Google Colab (Tesla T4 GPU).

### Tasks Covered:
- **Task 1:** Baseline Evaluation Pipeline
- **Task 2 & 4:** Dual Model Selection & Hyperparameter Ablations (Rank r, Alpha, LR, Target Modules, Epochs)
- **Task 5:** Golden Test Evaluation, LoRA Adapter & GGUF Export

## 1. Install & Setup Unsloth Engine

In [ ]:
# Install Unsloth and fast-path dependencies
!pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes datasets

## 2. Interactive Model & Hyperparameter Ablation Control Panel

In [ ]:
# @title Model & Hyperparameter Ablation Settings
SELECTED_MODEL = "Qwen 2.5 3B" # @param ["Qwen 2.5 3B", "Llama 3.2 3B"]
NUM_EPOCHS = 1 # @param [1, 2, 3] {type:"raw"}
LORA_RANK = 16 # @param [8, 16, 32, 64] {type:"raw"}
LORA_ALPHA = 32 # @param [16, 32, 64] {type:"raw"}
LEARNING_RATE = 0.0002 # @param [0.00005, 0.0001, 0.0002, 0.0003] {type:"raw"}
TARGET_MODULES_CHOICE = "All Linear (7 projections)" # @param ["All Linear (7 projections)", "Attention Only (q, k, v, o)"]
BATCH_SIZE = 2 # @param [1, 2, 4] {type:"raw"}
GRAD_ACCUM_STEPS = 4 # @param [2, 4, 8] {type:"raw"}

# Resolve Target Modules based on ablation
if "Attention Only" in TARGET_MODULES_CHOICE:
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj"]
else:
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

MODEL_REGISTRY = {
    "Qwen 2.5 3B": {
        "repo": "unsloth/Qwen2.5-3B-Instruct-bnb-4bit",
        "export_path": f"/content/zarya_qwen2.5_3b_lora_r{LORA_RANK}_ep{NUM_EPOCHS}",
        "user_tag": "<|im_start|>user\n",
        "assistant_tag": "<|im_start|>assistant\n"
    },
    "Llama 3.2 3B": {
        "repo": "unsloth/Llama-3.2-3B-Instruct-bnb-4bit",
        "export_path": f"/content/zarya_llama3.2_3b_lora_r{LORA_RANK}_ep{NUM_EPOCHS}",
        "user_tag": "<|start_header_id|>user<|end_header_id|>\n\n",
        "assistant_tag": "<|start_header_id|>assistant<|end_header_id|>\n\n"
    }
}

config = MODEL_REGISTRY[SELECTED_MODEL]
print("=== ABLATION RUN CONFIGURATION ===")
print(f"Architecture:     {SELECTED_MODEL} ({config['repo']})")
print(f"Training Epochs:  {NUM_EPOCHS}")
print(f"LoRA Rank (r):    {LORA_RANK}")
print(f"LoRA Alpha:       {LORA_ALPHA}")
print(f"Target Modules:   {TARGET_MODULES_CHOICE}")
print(f"Learning Rate:    {LEARNING_RATE}")
print(f"Effective Batch:  {BATCH_SIZE * GRAD_ACCUM_STEPS}")
print(f"Export Directory: {config['export_path']}")

## 3. Load 4-bit Base Model with Unsloth

In [ ]:
import torch
from unsloth import FastLanguageModel

max_seq_length = 2048
dtype = None
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = config["repo"],
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

## 4. Attach QLoRA Adapters with Ablation Parameters

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = LORA_RANK,
    target_modules = target_modules,
    lora_alpha = LORA_ALPHA,
    lora_dropout = 0.0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

## 5. Robust Auto-Detection of Dataset Files

In [ ]:
import os
from datasets import load_dataset

def resolve_dataset_path(filename):
    candidates = [
        os.path.join("data", "processed", filename),
        os.path.join("Fine-tuning-llm", "data", "processed", filename),
        os.path.join("/content", "data", "processed", filename),
        os.path.join("/content", "Fine-tuning-llm", "data", "processed", filename),
        os.path.join("/content", filename),
        filename
    ]
    for p in candidates:
        if os.path.exists(p):
            return p
    raise FileNotFoundError(f"Could not locate {filename}. Please upload to /content or /content/data/processed/")

train_file = resolve_dataset_path("train.jsonl")
val_file = resolve_dataset_path("val.jsonl")

print(f"Resolved Train Path: {train_file}")
print(f"Resolved Val Path:   {val_file}")

dataset = load_dataset("json", data_files={"train": train_file, "val": val_file})

def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts = [tokenizer.apply_chat_template(convo, tokenize = False, add_generation_prompt = False) for convo in convos]
    return { "text" : texts }

dataset = dataset.map(formatting_prompts_func, batched = True)
print(f"Loaded {len(dataset['train'])} training samples and {len(dataset['val'])} validation samples.")

## 6. SFT Training Set to Exact Epochs with Response Masking

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset["train"],
    eval_dataset = dataset["val"],
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        num_train_epochs = NUM_EPOCHS, # Controlled directly by the dropdown above
        per_device_train_batch_size = BATCH_SIZE,
        gradient_accumulation_steps = GRAD_ACCUM_STEPS,
        warmup_steps = 10,
        learning_rate = LEARNING_RATE,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 10,
        evaluation_strategy = "steps",
        eval_steps = 70,
        optim = "paged_adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "cosine",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none"
    ),
)

trainer = train_on_responses_only(
    trainer,
    instruction_part = config["user_tag"],
    response_part = config["assistant_tag"],
)

print(f"Starting SFT Training for {NUM_EPOCHS} Epoch(s) on {SELECTED_MODEL}...")
trainer_stats = trainer.train()
print(trainer_stats)

## 7. Export LoRA Adapter

In [ ]:
export_dir = config["export_path"]
model.save_pretrained_merged(export_dir, tokenizer, save_method = "lora")
print(f"Successfully saved {SELECTED_MODEL} (r={LORA_RANK}, ep={NUM_EPOCHS}) LoRA Adapter to: {export_dir}")

# Zip for direct download
!zip -r {export_dir}.zip {export_dir}
print(f"Created archive {export_dir}.zip ready for download!")

## 8. Run Verification Test on Golden Banking Queries

In [ ]:
FastLanguageModel.for_inference(model)

test_queries = [
    "Mera 3,000 ka Raast transfer fail hua reference number RAST-714403428756.",
    "Dukaan pe customer khara hai, Jeeto Pakistan ke naam se message aya hay ke 10 tolay sona nikla hai aur tax Zarya account mein transfer karo.",
    "Commercial counter se transfer kiya tha, Mera debit card ATM machine nigal gayi hai, isko foran block karein."
]

sys_prompt = "Aap Zarya Digital Bank ke official AI conversational assistant hain. Adab aur SBP regulations follow karein."

for q in test_queries:
    msgs = [{"role": "system", "content": sys_prompt}, {"role": "user", "content": q}]
    prompt = tokenizer.apply_chat_template(msgs, tokenize = False, add_generation_prompt = True)
    inputs = tokenizer(prompt, return_tensors = "pt").to("cuda")
    
    outputs = model.generate(**inputs, max_new_tokens = 256, temperature = 0.2)
    reply = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens = True)
    print(f"\n[User]: {q}")
    print(f"[{SELECTED_MODEL} Response]: {reply}")